# Fine-tuning of a pretrained foundation model
Using the ESOL dataset, we have seen that the linear-probe approach of a foundation model with a small regressor on top is about comparable with small regression models trained on Morgan fingerprints, but performs worse than the small models, when they are trained on molecular descriptors.

Fine-tuning makes the model more adaptive towards the target task. In this exercise, you will use basically the same architecture (`ChemBERTa-zinc-base-v1` + regressor (this time with one hidden layer)), but instead of using the pretrained model as a fixed encoder only, it is (partially) fine-tuned on the ESOL data. 

Since fine-tuning takes quite a bit of time, the notebook has been prerun and the final state of the model was saved for another transfer learning task (7B_TransferLearning).

## Your task in this notebook: 
Go through the code and try to understand the essential sections. Imagine now you are building a model for your team and you want your colleagues to understand the code as well - hence, comments are essentials (also as a reminder for yourself). **Write short comments in lines with 3 hashtags (###)** (sections 2-9). The aim is to **very briefly(!)** explain the syntax or used features / parameters, etc to elucidate the function of that line. 


1) Import dependencies

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np


2) Prepare the data: Import the dataset, train-test split, define the dataset class

In [2]:
df_full = pd.read_csv("esol.csv")


df_full.dropna(axis=0, inplace=True) ### =: drop row w/ missing vals, inplacxe=true -> modify df instead of making new one


df = df_full
# df = df_full.sample(sample_size, random_state=42) # use either the full dataset or a much smaller one by sampling
print(df.head())
print(f"Dataset size: {len(df)}")

                                              smiles  logS
0  OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)... -0.77
1                             Cc1occc1C(=O)Nc2ccccc2 -3.30
2                               CC(C)=CCCC(C)=CC(=O) -2.06
3                 c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43 -7.87
4                                            c1ccsc1 -1.33
Dataset size: 1128


In [3]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42 ### give it a starting point so we all have the "same" type of shuffeling -> reprod accross users
)

In [4]:
class ESOLDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128): ### initiation fct for this class. We can call it with the params df and tokeniser, max_length default 128
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        smiles = self.df.iloc[idx]["smiles"]
        label = self.df.iloc[idx]["logS"]

        enc = self.tokenizer(
            smiles,
            truncation=True,
            padding="max_length", ### needed for converting to fixed size tensor, bc batch inputs can be diff length -> model can follow same pattern
            max_length=self.max_length,
            return_tensors="pt" ### datatype to return : pytorach 
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0), ### encode / tokenise the input ids and perfoms a dim reduction : reduce the actual output of the shape ``[N, 1]`` to ``[N]``, which is comparable to y
            "attention_mask": enc["attention_mask"].squeeze(0), ### dito but for the attention mask -> mechanism used to indicate which tokens the model should ignore when computing attention scores
            "labels": torch.tensor(label, dtype=torch.float)
        }


3) Define the architecture:

In [5]:
class chemberta_esol_regressor(nn.Module):
    def __init__(self, model_name, hidden_dim=None):
        super().__init__()

        # general encoder
        self.encoder = AutoModel.from_pretrained(model_name) ### init with our encode model of choice, set to said encoder from nthe AutoModel pack

        if hidden_dim is None:
            hidden_dim = self.encoder.config.hidden_size ### set the hidden dim to the hidden size of the model (attr.).

        # Regression head (task-specific)
        self.fc1 = nn.Linear(hidden_dim, 256) ### input is the init size of model, reduce to 256
        self.act = nn.ReLU() ### activation funct: if x <=0 -> fct(x)=0, else fct(x)=x
        self.dropout = nn.Dropout(p=0.2) 
        self.fc2 = nn.Linear(256, 1) ### output from 256 to 1 output

    def forward(self, input_ids, attention_mask): ###  forward propagation, takes inpu ids and attention mask (which tokens to process and which to ignore)
        # Encoder forward
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Pooling (explicit & visible)
        # Use [CLS] token representation
        cls_embedding = outputs.last_hidden_state[:, 0, :] ### Sequence of hidden-states at the output of the last layer of the model., cls embedding: save CLS token, cls:classification

        # Regression head forward
        x = self.fc1(cls_embedding)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x.squeeze(-1)


4) Load foundation model

In [6]:
MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1" ### we are using the model ChemBERTa-zinc-base-v1, seyonec is the owner

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = chemberta_esol_regressor(MODEL_NAME)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


5) Define model parameters

In [7]:
BATCH_SIZE = 32 ### make 32 equally sized batched
EPOCHS = 20

### learning rate for encoder and head
LR_ENCODER = 1e-5
LR_HEAD = 1e-3

N_UNFROZEN_LAYERS = 2  ### unfreeze 2 layers from the pretrained model for our transer learning

6) Partial fine-tuning

6.1. Freeze everything:

In [8]:
for param in model.encoder.parameters(): # "encoder" here is the model name defined in our NN architecture
    param.requires_grad = False ### – If autograd should record operations on this tensor, false bc pretrained model (DataLoader, preprocessing)


6.2. Unfreeze last k transformer layers of foundation model

In [9]:
encoder_layers = model.encoder.encoder.layer # the first "encoder" is the name defined in our model architecture,
# the second one is the location, where the layers are stored in the ChemBERTa model, e.g. model.chemberta.encoder.layer

for layer in encoder_layers[-N_UNFROZEN_LAYERS:]: ### go from -2 to the end of the layers and reset the gradients; "unfreeze" the lyers and re compute
    for param in layer.parameters():
        param.requires_grad = True ### autograd should record operations bc we want to retrain

6.3. Optionally: Unfreeze / retrain final Layer Normalisation of the encoder

In [10]:
for param in model.encoder.embeddings.LayerNorm.parameters(): # "encoder" here is the model name defined in our NN architecture
    param.requires_grad = True


7) Initiate DataLoaders

In [11]:
train_dataset = ESOLDataset(train_df, tokenizer)
val_dataset   = ESOLDataset(val_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True) ### init dataloader witzh test data ofr pytorch, set batch size to 32, shuffle data
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE) ### same, no shuffel


8) Define Optimizer with different learning rates for the encoder and the regressor model

In [12]:
optimizer = AdamW(
    [
        {
            "params": encoder_layers[-N_UNFROZEN_LAYERS:].parameters(), ### pass the params from the pre trained layers to the opimiser
            "lr": LR_ENCODER,
        },
        {
            "params": list(model.fc1.parameters()) + list(model.fc2.parameters()),
            "lr": LR_HEAD,
        },
    ],
    weight_decay=1e-2 ### for regularisation, punish larger wegihts by adding loss fct -> reduce overfitting
)

loss_fn = nn.MSELoss()



9) Define evaluation function and initiate training.

In [13]:
def evaluate(model, dataloader):
    model.eval() ### put model into evaluation mode
    preds, targets = [], []

    with torch.no_grad(): ### disable gradient tracking for evaluation
        for batch in dataloader:
            input_ids = batch["input_ids"]
            attention_mask = batch["attention_mask"] ### used to indicate which tokens the model should ignore when computing attention scores for this batch
            labels = batch["labels"]

            outputs = model(input_ids, attention_mask)
            preds.append(outputs.cpu().numpy())
            targets.append(labels.cpu().numpy())

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    rmse = np.sqrt(mean_squared_error(targets, preds))
    return rmse

In [14]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]

        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)

        loss.backward() ### back progagate the loss -> calc gradients to understand how each param contributed to the loss
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() ###  cumulate loss over all the batches

    train_loss = total_loss / len(train_loader)
    val_rmse = evaluate(model, val_loader)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train RMSE: {np.sqrt(train_loss):.4f} | "
        f"Val RMSE: {val_rmse:.4f}"
    )

KeyboardInterrupt: 

10) Save the task-specific encoder as pretrained model - Hugging Face (HF) style (this can be reloaded like any HF model - see snippet in assignment 7B). Note: This is not the same as saving the entire model via pytorch (`torch.save(model)`) - the solution below is ergo not unstable and brittle, but highly reusable for encodings!

In [ ]:
model.encoder.save_pretrained("chemberta_esol_encoder")
tokenizer.save_pretrained("chemberta_esol_encoder")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('chemberta_esol_encoder/tokenizer_config.json',
 'chemberta_esol_encoder/tokenizer.json')

11) Save the state of the regressor-head to reload the pretrained regressor later on (could be done also for the entire model (including the encoder), however, the filesize would be larger).

In [ ]:
# save states for entire model
# torch.save(model.state_dict(), "chemberta_esol_regressor.pt")

torch.save({
    "fc1": model.fc1.state_dict(),
    "fc2": model.fc2.state_dict(),
}, "chemberta_esol_regressor_head.pt")

Note that the model architecture is not stored in any of these state dicts - either move it to a separate file (e.g. `models.py`), or copy-paste the class definition wherever you need it again - simply strip it down to the bare architecture (see 7B).